# Week 6 — Evaluation

**Focus:** How do dependence, search, and costs change the strength of forecast evidence?

*Live-coding notebook*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/smc77/uc_finmlai/blob/main/lectures/week06/week6_demos.ipynb)

## What this notebook is about

A small forecasting edge can matter when it repeats across genuinely distinct opportunities, but repetition, costs, search, and changing regimes all affect how much evidence we really have. This notebook asks how to turn forecast performance into a well-supported conclusion.

The demonstrations move from cross-sectional rank information and decile spreads to breadth, gross-to-net performance, multiple testing and the deflated Sharpe ratio, changing model capacity, and time-varying states. Optional state-space and hidden-state examples show when a latent regime is recoverable and when the data do not identify it.

By the end, you should be able to:

- distinguish the number of rows from the number of meaningfully independent opportunities;
- connect rank information, breadth, turnover, and net performance;
- record the breadth of a model or strategy search;
- treat window length and model capacity as selected choices; and
- state what a latent-state estimate does not prove.

These are controlled experiments and sensitivity analyses. Their purpose is to calibrate evidence, not to advertise a strategy.

### Course context

For a concise review and the optional textbook route, see [Week 6 summary](https://github.com/smc77/uc_finmlai/blob/main/lectures/week06/week6_summary.md) and [Week 6 reading guide](https://github.com/smc77/uc_finmlai/blob/main/lectures/week06/reading.md). The notebook itself is designed to remain understandable without them.

## How to use this notebook

At the first break, run any **Setup** cell and then the **Imports** cell. If there is no Setup cell, begin with Imports. After that, use the slide cue to jump to the named demo; you do not need to rerun the whole notebook at every break. To check the whole notebook from a clean start, use **Runtime → Run all**.

Each demo follows the same rhythm: predict what the output should show, run the cell, and follow the task immediately underneath it. An **After you run** cell is editable: double-click it, replace `[write here]`, and press Shift-Enter. When an editable research-record entry appears, its completed comparison sits below it in a collapsed box—write your version before opening that box. This notebook is generated from the same source code the lecturer runs live.

## Jump to a demonstration

Both links jump within *this* notebook — neither opens a new tab. Use **Colab** when the notebook is open in Colab, **Jupyter** when it is open in Jupyter or rendered to HTML. You do not need to scroll through or rerun the whole notebook.

> If the **Jupyter** links do nothing in JupyterLab or Notebook 7, that is the windowed renderer, not a broken link: cells outside the viewport are not in the page, so there is no anchor to scroll to. Set *Settings → Settings Editor → Notebook → Windowing mode* to `defer` (or `none`) and they work.

### Deck A

<ul>
<li>Demo 1 — Per-period rank IC and IC-IR on a simulated cross-section — <a href="#Demo-1-%E2%80%94-Per-period-rank-IC-and-IC-IR-on-a-simulated-cross-section">Jupyter</a> · <a href="#scrollTo=md-866d627c647b">Colab</a></li>
<li>Demo 2 — Monotonic decile spread — <a href="#Demo-2-%E2%80%94-Monotonic-decile-spread">Jupyter</a> · <a href="#scrollTo=md-80799bca3b2b">Colab</a></li>
<li>Real-data companion — industry momentum rank IC, spread, and turnover — <a href="#Real-data-companion-%E2%80%94-industry-momentum-rank-IC,-spread,-and-turnover">Jupyter</a> · <a href="#scrollTo=md-bfd230f93c6a">Colab</a></li>
<li>Demo 3 — Fundamental law: IR ≈ IC × sqrt(breadth) — <a href="#Demo-3-%E2%80%94-Fundamental-law%3A-IR-%E2%89%88-IC-%C3%97-sqrt(breadth)">Jupyter</a> · <a href="#scrollTo=md-e5b865c586f1">Colab</a></li>
<li>Demo 4 — Gross-to-net Sharpe under transaction costs — <a href="#Demo-4-%E2%80%94-Gross-to-net-Sharpe-under-transaction-costs">Jupyter</a> · <a href="#scrollTo=md-8ef3a0ad8aa6">Colab</a></li>
<li>Demo 5 — Multiple-testing mirage — <a href="#Demo-5-%E2%80%94-Multiple-testing-mirage">Jupyter</a> · <a href="#scrollTo=md-6d40683d61aa">Colab</a></li>
</ul>

### Deck B

<ul>
<li>Demo 6 — Move the interpolation peak — <a href="#Demo-6-%E2%80%94-Move-the-interpolation-peak">Jupyter</a> · <a href="#scrollTo=md-cacb0ce99b2c">Colab</a></li>
<li>Demo 7 — Sweep one training-window dimension — <a href="#Demo-7-%E2%80%94-Sweep-one-training-window-dimension">Jupyter</a> · <a href="#scrollTo=md-0721311e62d2">Colab</a></li>
</ul>

### Optional frontier

<ul>
<li>Demo 8 — Kalman filter for time-varying beta — <a href="#Demo-8-%E2%80%94-Kalman-filter-for-time-varying-beta">Jupyter</a> · <a href="#scrollTo=md-cfaa9ca476a3">Colab</a></li>
<li>Demo 9 — A 2-state Gaussian-emission HMM (recoverable case) — <a href="#Demo-9-%E2%80%94-A-2-state-Gaussian-emission-HMM-(recoverable-case)">Jupyter</a> · <a href="#scrollTo=md-ee9849063e29">Colab</a></li>
<li>Demo 10 — §7 knob-dial: when is a regime recoverable? — <a href="#Demo-10-%E2%80%94-%C2%A77-knob-dial%3A-when-is-a-regime-recoverable%3F">Jupyter</a> · <a href="#scrollTo=md-5a557801a8b6">Colab</a></li>
</ul>

### Textbook companion

<ul>
<li>Overlapping-returns companion — the same regression the textbook runs — <a href="#Overlapping-returns-companion-%E2%80%94-the-same-regression-the-textbook-runs">Jupyter</a> · <a href="#scrollTo=md-49577bc6ee75">Colab</a></li>
</ul>

## Read the code: follow one comparison at a time

A score answers a question, not every question. First identify the input, the operation, and what the output measures. Demos 1–7 support four lecture breaks. Demos 8–10 are optional state-model extensions; the real-data and overlapping-return companions are also optional. Run cells in order because later cells reuse earlier arrays and helper functions.

## Project research-record checkpoint

This is the first project-phase notebook. Copy `project/RESEARCH_RECORD_TEMPLATE.md` to the root of your own project repository, name it exactly `RESEARCH_RECORD.md`, and commit the initialized file. **Do not copy the project record into this notebook.** Use this week's exercises to make these project fields concrete where applicable:

1. **Identity:** question, intended decision, owner, repository, and current commit.
2. **Information contract:** decision time, feature cutoff, outcome interval, data source and vintage, sample dates, and units.
3. **Baseline and test:** feasible baseline, time-aware development/selection/final-test roles, primary metric, and denominator.
4. **Search:** candidates tried or planned, breadth or dependence assumptions, and the rule for counting trials.
5. **Claim boundary:** costs or frictions, one important limitation, and the narrow claim the design could warrant.

### Four recorded-lecture breaks

1. **Rank and diagnose — Demos 1–2:** compute per-date rank IC and a decile spread from the same scores; separate ordering evidence from a net decision result.
2. **Scale, implement, and search — Demos 3–5:** calibrate the independent-breadth approximation, carry one score through two cost ledgers, calculate a search-adjusted Sharpe statistic, and test the development-selected noise strategy later.
3. **Interpolate — Demo 6:** vary a declared control and distinguish a second descent from evidence that the high-capacity endpoint is best.
4. **Stress memory — Demo 7:** hold outer dates fixed while window length changes; compare the window selected by predictive loss with the one selected by turnover.

Demos 8–10 accompany the optional state-space and HMM frontier reading; they are not additional recorded-lecture breaks.

Append a dated decision-log entry whenever this week's evidence changes the project design. Mark fields `not applicable yet` rather than inventing them.

### Imports

> **Run this once before any demo.** It loads the packages and helper functions used below; there is no result to interpret in this cell.

In [ ]:
from io import BytesIO, StringIO
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, norm

rng = np.random.default_rng(42)


def save_lecture_figure(filename):
    """Save the current notebook figure beside the Week 6 lecture figures."""
    if "__file__" in globals():
        figure_dir = Path(__file__).resolve().parents[1] / "figures"
    else:
        # Jupyter and Colab kernels do not define ``__file__``. Prefer the
        # repository figure directory when the notebook is running locally,
        # and otherwise keep the generated artifact beside the notebook.
        figure_dir = next(
            (
                root / "lectures" / "week06" / "figures"
                for root in (Path.cwd(), *Path.cwd().parents)
                if (root / "lectures" / "week06" / "figures").is_dir()
            ),
            Path.cwd() / "figures",
        )
    output = figure_dir / filename
    output.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(output, dpi=180, bbox_inches="tight")


def annualized_sharpe(values, periods_per_year=252):
    """Annualized mean-to-volatility ratio under a declared frequency."""
    values = np.asarray(values, dtype=float)
    return values.mean() / values.std(ddof=1) * np.sqrt(periods_per_year)


def ewma(values, alpha):
    """A small explicit EWMA helper used by the implementation experiment."""
    result = np.empty_like(values, dtype=float)
    result[0] = values[0]
    for index in range(1, len(values)):
        result[index] = alpha * values[index] + (1 - alpha) * result[index - 1]
    return result


def course_csv(relative_path):
    """Read bundled Fama-French data locally, or its public source in Colab."""
    for root in (Path.cwd(), *Path.cwd().parents):
        local = root / relative_path
        if local.exists():
            return pd.read_csv(local), str(local)
    archives = {
        "datasets/famafrench/ff_factors_daily.csv": "F-F_Research_Data_Factors_daily_CSV.zip",
        "datasets/famafrench/ff_12industry_daily.csv": "12_Industry_Portfolios_daily_CSV.zip",
    }
    archive = archives[relative_path]
    url = f"https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/{archive}"
    with urlopen(url) as response, ZipFile(BytesIO(response.read())) as zipped:
        lines = zipped.read(zipped.namelist()[0]).decode("utf-8").splitlines()
    header_row = next(i for i, line in enumerate(lines) if line.startswith(","))
    rows = []
    for line in lines[header_row + 1:]:
        first = line.split(",", 1)[0].strip()
        if len(first) == 8 and first.isdigit():
            rows.append(line)
        elif rows:
            break
    frame = pd.read_csv(StringIO("\n".join([lines[header_row], *rows])))
    return frame.rename(columns={frame.columns[0]: "date"}), url

### Demo 1 — Per-period rank IC and IC-IR on a simulated cross-section

> **Break cue:** Deck A, after recording segment S1. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

### One date produces one ranking comparison

**Purpose:** check whether higher forecasts tend to precede higher returns. `pred` and `ret` each have shape `(252, 200)`: dates by assets. `pred[t]` selects all 200 scores on one date. `spearmanr(pred[t], ret[t]).statistic` compares their ranks; it does not fit a forecasting model. The simulation supplies the scores and a weak stable relationship.

**Worked step:** predicted order `[1, 2, 3]` and realized order `[3, 2, 1]` have correlation −1. Identical orders have correlation +1. The loop makes 252 correlations, and `ics.mean()` summarizes them. `mean / std * sqrt(252)` is the annualized IC information ratio under the independent-daily convention here, not a portfolio Sharpe or automatic significance test. The code uses population-form `std()` consistently for this descriptive display.

**Predict, then check:** removing `latent_alignment` should move mean IC toward zero across repeated seeds. It need not equal zero in one sample. Read the bar chart as a sequence of date-level comparisons, not 50,400 independent experiments.

In [ ]:
T, N = 252, 200          # 1 yr daily, 200 assets
latent_alignment = 0.05  # correlation with the latent return component
demo1_rng = np.random.default_rng(601)
# Generate target as a noisy view of a true score
true = demo1_rng.standard_normal((T, N))
noise = demo1_rng.standard_normal((T, N))
# Mix to control IC: pred = a*true + b*noise
a = latent_alignment; b = np.sqrt(1 - a**2)
pred = a * true + b * noise
ret = true * 0.01 + demo1_rng.standard_normal((T, N)) * 0.01  # actual realized

ics = np.array([spearmanr(pred[t], ret[t]).statistic for t in range(T)])
mean_ic = ics.mean()
ic_ir = mean_ic / ics.std() * np.sqrt(252)
print(f"Mean rank IC: {mean_ic:+.4f}")
print(f"Annualized IC-IR: {ic_ir:.2f}")

plt.figure(figsize=(8, 3))
plt.bar(np.arange(T), ics, color=["tab:green" if v > 0 else "tab:red" for v in ics],
        alpha=0.5)
plt.axhline(mean_ic, color="k", lw=1)
plt.axhline(mean_ic + ics.std(), color="k", ls=":", lw=0.7)
plt.axhline(mean_ic - ics.std(), color="k", ls=":", lw=0.7)
plt.title(f"Per-period rank IC (mean={mean_ic:+.3f}, IC-IR≈{ic_ir:.1f})")
plt.tight_layout(); save_lecture_figure("demo1_rank_ic.png"); plt.show()
# Expected: mean rank IC ~0.03. The annualized IC-IR looks implausibly high because the
#           simulation repeats independent dates with a stable planted relation; it is not a market forecast.

#### After you run

Record the mean rank IC, the annualized IC-IR, and the unit at which IC was
sampled. Then explain why this deliberately stable simulation can produce an
IC-IR that would be implausibly strong as an empirical market conclusion.

**Your notes:** `[write here]`

### Demo 2 — Monotonic decile spread

> **Break cue:** Deck A, after recording segment S1. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

### Turn a ranking into a simple portfolio

**Purpose:** show what ordering would imply for a simple action. `argsort` returns asset indices from lowest to highest score. `array_split(order, 10)` makes ten groups of 20; `ret[t, bins[d]].mean()` is one group's equal-weight return that day. No new model is fitted.

**Worked step:** if the high group's mean is 0.002 and the low group's is −0.001, the long-minus-short spread is 0.003, or 30 basis points. `long_short` is a time series, so its mean-to-standard-deviation ratio measures portfolio returns, unlike the date-level IC in Demo 1. The two unit-sized legs imply gross exposure two; borrowing, financing, drift, and costs are absent.

**Predict, then check:** a small positive IC need not make every neighboring bar rise. Look at all ten bars before claiming monotonicity. The planted signal is known from the data generator; a rising sample chart alone would not prove a real market signal.

In [ ]:
decile_returns = np.zeros((T, 10))
for t in range(T):
    order = np.argsort(pred[t])
    bins = np.array_split(order, 10)
    for d in range(10):
        decile_returns[t, d] = ret[t, bins[d]].mean()
avg_per_decile = decile_returns.mean(axis=0)

plt.figure(figsize=(8, 3.5))
plt.bar(np.arange(10) + 1, avg_per_decile)
plt.xlabel("prediction decile (1=low, 10=high)")
plt.ylabel("mean realized return")
plt.title("Decile means from a planted weak ranking relation")
plt.tight_layout(); save_lecture_figure("demo2_decile_spread.png"); plt.show()

long_short = decile_returns[:, 9] - decile_returns[:, 0]
sharpe_gross = long_short.mean() / long_short.std() * np.sqrt(252)
print(f"Top-minus-bottom gross Sharpe (annualized): {sharpe_gross:.2f}")
print("Reminder: GROSS. Costs and turnover come next.")

#### After you run

Describe the shape across all ten deciles and record the gross top-minus-bottom
Sharpe. State what this diagnostic adds beyond IC and what it still omits before
the spread could be described as a net investment result.

**Your notes:** `[write here]`

### Mini-project: compare ranking with a diagnostic decision

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

Run **Demos 1–2**. Compute per-date IC and a top-minus-bottom spread from the same scores. Explain
what the spread adds—and what is still missing from a net result.

**Research record:** the per-date IC series and its mean; the decile spread and its monotonicity; what the spread adds beyond ordering, and what a net result still requires.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Mini-project: grade one forecast three ways

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

**Notebook:** run **Demos 1, 2, and 4 — Rank IC, decile spread, and gross-to-net Sharpe**.

Compute per-date IC, a top-minus-bottom spread, and net return after a
cost schedule. Explain why the three answers differ.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Real-data companion — industry momentum rank IC, spread, and turnover

> **Optional empirical comparison:** Deck A, after recording segment S1. Run this after the known-truth demo. Use the same transformation and compare the simulation with the historical series. Record one important difference and the most likely reason for it.

### The historical example needs an explicit clock

**Purpose:** repeat the ranking and cost ledger on 12 industry portfolios. CSV returns are percentages; dividing by 100 converts them to decimals. `rolling(63).sum()` is a trailing sum of daily returns, not a compounded return. `shift(-1)` aligns tomorrow's outcome with today's score; it creates a target, never an input feature.

The three highest industries receive weights +1/3 and the three lowest −1/3. `sum(weights * next_returns, axis=1)` produces a daily gross spread. `abs(diff(weights)).sum(axis=1)` charges the sum of absolute target changes, with initial entry included. For a 10-point sale and 10-point purchase that convention counts 0.20, not half that number. The cost rate must match the convention.

**Limit:** scores use the close at t and returns start at that same close. This is a descriptive study; an executable strategy needs a cutoff and attainable execution prices. The turnover proxy omits price drift, and costs omit borrow, financing, and impact. The textbook figure uses monthly data and an 11-month score, so these numbers need not match it.

**Try:** change only the cost rate. IC and gross returns should remain identical; net returns should change.

In [ ]:
industry_raw, industry_source = course_csv("datasets/famafrench/ff_12industry_daily.csv")
industry_raw["date"] = pd.to_datetime(industry_raw["date"].astype(str), format="%Y%m%d")
industry_returns = industry_raw.set_index("date").sort_index().loc["1990":] / 100.0
industry_score = industry_returns.rolling(63).sum()
industry_next = industry_returns.shift(-1)
industry_mask = industry_score.notna().all(axis=1) & industry_next.notna().all(axis=1)
industry_score = industry_score.loc[industry_mask]
industry_next = industry_next.loc[industry_mask]

industry_ic = pd.Series(
    [spearmanr(industry_score.loc[date], industry_next.loc[date]).statistic
     for date in industry_score.index],
    index=industry_score.index,
)
industry_weights = np.zeros_like(industry_score.to_numpy())
for row, values in enumerate(industry_score.to_numpy()):
    order = np.argsort(values)
    industry_weights[row, order[:3]] = -1 / 3
    industry_weights[row, order[-3:]] = 1 / 3
industry_gross = np.sum(industry_weights * industry_next.to_numpy(), axis=1)
industry_turnover = np.r_[np.abs(industry_weights[0]).sum(),
                          np.abs(np.diff(industry_weights, axis=0)).sum(axis=1)]
industry_net = industry_gross - 5e-4 * industry_turnover

print("Source: Kenneth R. French Data Library, bundled 12 industry portfolios")
print(f"File: {industry_source}")
print(f"Sample: {industry_score.index.min().date()} through {industry_score.index.max().date()}")
print("Units: decimal daily returns; score is trailing 63-session return through t; target is t+1")
print(f"Mean daily cross-sectional rank IC: {industry_ic.mean():+.4f}")
print(f"Top-three minus bottom-three gross Sharpe: {annualized_sharpe(industry_gross):+.2f}")
print(f"Mean sum-absolute target-change proxy: {industry_turnover.mean():.2f}")
print(f"Net Sharpe at 5 bps per unit turnover: {annualized_sharpe(industry_net):+.2f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(industry_ic.rolling(252).mean(), lw=0.8)
axes[0].axhline(0, color="0.3", lw=0.5)
axes[0].set_title("One-year rolling mean rank IC")
axes[1].plot(industry_score.index, np.cumsum(industry_gross), label="gross")
axes[1].plot(industry_score.index, np.cumsum(industry_net), label="net of 5 bps")
axes[1].set_title("A real spread needs a cost ledger")
axes[1].legend()
fig.tight_layout(); plt.show()
print("Twelve industry portfolios are not twelve independent bets; the IC-IR would need dependence-aware inference.")
print("Illustrative clock: score through close t, return from close t to close t+1; an executable study needs a tradable cutoff and prices.")
print("Target-change turnover ignores holding drift, borrow, financing, and impact. This daily 63-session score differs from the monthly textbook example.")

#### After you run

Record the source, sample dates, mean IC, gross spread Sharpe, turnover, and net
Sharpe. Compare the historical path with the known-truth simulation and give
one reason the empirical result is less stable.

**Your notes:** `[write here]`

### Demo 3 — Fundamental law: IR ≈ IC × sqrt(breadth)

> **Break cue:** Deck A, after recording segment S2. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

### Many separate weak opportunities can add up

**Purpose:** isolate breadth without trading costs or correlated names. `forecast` and `outcome` have shape `(4000, B)`. The generator plants Pearson correlation 0.05 for each bet. Multiplying them makes each bet's payoff, and `mean(..., axis=1)` averages the B payoffs within each period. This is not the Spearman IC from Demo 1.

**Worked step:** 0.05 times the square root of 100 gives an idealized ratio of 0.50. For this Gaussian product-payoff construction, the exact population ratio is `0.05 * sqrt(B) / sqrt(1 + 0.05**2)`; the familiar law is a close small-correlation approximation. The printed ratios are per simulated period, not annualized daily Sharpes.

**Predict:** copying one bet into 100 columns would not create 100 independent opportunities. The demo deliberately supplies independence and frictionless transfer; actual effective breadth needs a dependence model.

In [ ]:
pearson_ic_per_bet = 0.05
breadths = [1, 5, 25, 100, 400, 1600]
realized_irs, theoretical = [], []
demo3_rng = np.random.default_rng(603)
n_periods = 4000
for B in breadths:
    forecast = demo3_rng.standard_normal((n_periods, B))
    outcome = (pearson_ic_per_bet * forecast
               + np.sqrt(1 - pearson_ic_per_bet**2)
               * demo3_rng.standard_normal((n_periods, B)))
    active_return = np.mean(forecast * outcome, axis=1)
    realized_irs.append(active_return.mean() / active_return.std(ddof=1))
    theoretical.append(pearson_ic_per_bet * np.sqrt(B))

print("Breadth | realized IR | Pearson IC*sqrt(B)")
for B, r, t in zip(breadths, realized_irs, theoretical):
    print(f"  {B:5d}  | {r:9.3f}   | {t:7.3f}")
# Expected: the two columns roughly track.

#### After you run

Compare the simulated information ratios with `IC × sqrt(breadth)`. Name two
assumptions that would make nominal breadth overstate attainable performance in
a real financial application.

**Your notes:** `[write here]`

### Demo 4 — Gross-to-net Sharpe under transaction costs

> **Break cue:** Deck A, after recording segment S2. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

### Recompute returns after costs, then recompute Sharpe

**Purpose:** compare acting on every noisy score with smoothing the same score. `tanh` bounds raw positions; `ewma` averages past and current positions with weight `alpha=0.08` on the newest value. A smaller alpha reacts more slowly. The simulation's hidden state persists, so smoothing can remove noise; that advantage is not universal.

`abs(diff(weight))` measures target-position changes; the first entry charges entering the initial position. `gross = weight * next_return` and `net = gross - cost_bps / 10000 * trade` keep all quantities in decimal return units. Full-path root-mean-square position scaling standardizes this illustration; it is **not** estimated forecast volatility or a deployable sizing rule. A live comparison needs scaling estimated from past data and trades based on drifted pre-trade holdings.

**Worked step:** turnover 0.30 at 10 basis points costs `0.30 * 10 / 10000 = 0.0003`, or 3 basis points. Calculate Sharpe from the resulting net path; do not subtract 3 from gross Sharpe. Break-even cost divides mean gross return by mean turnover.

**Try:** hold the trades fixed and raise cost. Read where each net-Sharpe curve crosses zero. Do not refit or rescale at every cost, because that would introduce another search.

In [ ]:
demo4_rng = np.random.default_rng(604)
n_days = 3500
persistence = 0.96
state = np.empty(n_days)
state[0] = demo4_rng.standard_normal()
for day in range(1, n_days):
    state[day] = (persistence * state[day - 1]
                  + np.sqrt(1 - persistence**2) * demo4_rng.standard_normal())

score = state + 1.1 * demo4_rng.standard_normal(n_days)
next_return = 0.0005 * state + 0.01 * demo4_rng.standard_normal(n_days)
raw_weight = np.tanh(score)
implementations = {
    "slow": ewma(raw_weight, alpha=0.08),
    "fast": raw_weight,
}
cost_grid = np.linspace(0, 60, 61)
cost_rows, ledger_rows = [], []
for label, weight in implementations.items():
    # Full-path RMS scaling standardizes this teaching comparison; it is not
    # a live sizing estimate. A deployable rule must estimate scale from past data.
    weight = weight / np.sqrt(np.mean(weight**2))
    trade = np.r_[abs(weight[0]), np.abs(np.diff(weight))]
    gross = weight * next_return
    breakeven_bps = gross.mean() / trade.mean() * 10_000
    for cost_bps in cost_grid:
        net = gross - cost_bps / 10_000 * trade
        cost_rows.append({
            "implementation": label,
            "cost_bps": cost_bps,
            "net_sharpe": annualized_sharpe(net),
        })
    ledger_rows.append({
        "implementation": label,
        "gross_mean_bps": gross.mean() * 10_000,
        "gross_sharpe": annualized_sharpe(gross),
        "turnover": trade.mean(),
        "breakeven_bps": breakeven_bps,
        "net_sharpe_10bps": annualized_sharpe(gross - 10 / 10_000 * trade),
    })

cost_results = pd.DataFrame(cost_rows)
implementation_ledger = pd.DataFrame(ledger_rows).set_index("implementation")
print(implementation_ledger.round(3).to_string())

plt.figure(figsize=(7.5, 3.7))
for label, color in [("slow", "green"), ("fast", "red")]:
    rows = cost_results[cost_results.implementation == label]
    plt.plot(rows.cost_bps, rows.net_sharpe, color=color, label=label)
    plt.axvline(implementation_ledger.loc[label, "breakeven_bps"],
                color=color, ls=":", lw=1)
plt.axhline(0, color="k", lw=0.5)
plt.xlabel("cost per dollar traded (bps)")
plt.ylabel("net Sharpe (annualized)")
plt.title("One score, two implementations, explicit cost sensitivity")
plt.legend(); plt.tight_layout(); save_lecture_figure("demo4_cost_sensitivity.png"); plt.show()
# Expected: the fast implementation has much greater turnover and a much lower
#           break-even cost even though both begin from the same score.

#### After you run

Identify which implementation loses performance faster as cost rises and
explain why. Record the approximate break-even cost from the plot and state
which quantities were held fixed during the cost sweep.

**Your notes:** `[write here]`

### Demo 5 — Multiple-testing mirage

> **Break cue:** Deck A, after recording segment S2. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

### Search chooses the luckiest noise, then new data test it

**Purpose:** separate selection luck from planted skill. Both arrays have shape `(300, 2520)`: candidate strategies by days. Every population mean is zero. `argmax(development_sharpes)` selects one index using development only; `test_sharpes[selected]` evaluates that same index on an independent sample. The test is a separately drawn hypothetical later period, not dated market data.

`maximum.accumulate` records the best score seen as the search grows. The expected-best-null approximation uses all 300 independent attempts and the spread of their Sharpes. `norm.ppf` converts a percentile to a normal quantile. The deflated-Sharpe calculation compares the winner with that elevated benchmark, using sample skewness, non-excess kurtosis, and sample length. Divide annualized Sharpes by `sqrt(252)` before combining them with the daily observation count; `norm.cdf` then maps the standardized comparison to a value from zero to one.

**Interpretation:** this is an approximate search-adjusted confidence statistic under the declared assumptions, not proof of skill or a reduced Sharpe estimate. Dependent candidates need an effective-trial treatment; serially dependent returns need further care. An undocumented search cannot be recovered from the winner alone.

**Predict:** increasing the search size tends to raise the development winner. Its independent test score remains noisy around zero. Never choose a second winner from the test column.

In [ ]:
T = 252 * 10
N_strats = 300
demo5_rng = np.random.default_rng(605)
development = demo5_rng.standard_normal((N_strats, T)) * 0.01
test = np.random.default_rng(655).standard_normal((N_strats, T)) * 0.01
development_sharpes = (development.mean(axis=1) / development.std(axis=1, ddof=1)
                       * np.sqrt(252))
test_sharpes = (test.mean(axis=1) / test.std(axis=1, ddof=1)
                      * np.sqrt(252))
selected = int(np.argmax(development_sharpes))
best_so_far = np.maximum.accumulate(development_sharpes)

# In this controlled experiment the 300 trials are independent and fully logged.
# Estimate the expected best null Sharpe, then place that benchmark inside
# the probabilistic-Sharpe approximation used by the deflated Sharpe ratio.
cross_trial_sigma = development_sharpes.std(ddof=1)
expected_best_null = cross_trial_sigma * (
    (1 - np.euler_gamma) * norm.ppf(1 - 1 / N_strats)
    + np.euler_gamma * norm.ppf(1 - 1 / (N_strats * np.e))
)
selected_returns = development[selected]
standardized = ((selected_returns - selected_returns.mean())
                / selected_returns.std(ddof=0))
sample_skew = np.mean(standardized ** 3)
sample_kurtosis = np.mean(standardized ** 4)  # non-excess kurtosis
selected_sr_period = development_sharpes[selected] / np.sqrt(252)
null_sr_period = expected_best_null / np.sqrt(252)
dsr_denominator = np.sqrt(
    1 - sample_skew * selected_sr_period
    + (sample_kurtosis - 1) / 4 * selected_sr_period ** 2
)
deflated_sharpe = norm.cdf(
    (selected_sr_period - null_sr_period) * np.sqrt(T - 1)
    / dsr_denominator
)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].plot(np.arange(1, N_strats + 1), best_so_far)
axes[0].axhline(expected_best_null, color="tab:orange", ls="--",
                label="expected best null")
axes[0].set_xlabel("strategies tried (cumulative)")
axes[0].set_ylabel("best development Sharpe")
axes[0].set_title("The selected noise result rises with search")
axes[0].legend()
axes[1].bar(["development", "later test"],
            [development_sharpes[selected], test_sharpes[selected]],
            color=["tab:red", "tab:green"])
axes[1].axhline(0, color="k", lw=0.5)
axes[1].set_ylabel("annualized Sharpe")
axes[1].set_title("The selected index on untouched noise")
plt.tight_layout(); save_lecture_figure("demo5_selection_mirage.png"); plt.show()
print(f"Candidates searched: {N_strats}")
print(f"Selected development Sharpe: {development_sharpes[selected]:+.2f}")
print(f"Expected best null Sharpe: {expected_best_null:+.2f}")
print(f"Deflated Sharpe ratio: {deflated_sharpe:.2f}")
print(f"Same selected strategy, later Sharpe: {test_sharpes[selected]:+.2f}")
print("Every candidate has zero population skill by construction.")
# Expected: selection produces a positive development winner; its later result
#           is another draw around the zero planted truth.

#### After you run

Record the number of strategies searched, the selected development Sharpe, the
expected best null Sharpe, the deflated Sharpe ratio, and the same selected
index's later Sharpe. The true Sharpe of every candidate is zero. Explain what
the deflated statistic adds, why it depends on a complete search record, and why
the later value need not be exactly zero.

**Your notes:** `[write here]`

### Mini-project: apply costs to fixed trades

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

Run **Demos 3–5**. Compare nominal breadth with independent breadth, sweep one
cost parameter while preserving the trades, and calculate the deflated Sharpe
of a development-selected noise strategy before carrying that same strategy
into an untouched later sample.

**Research record:** nominal against effective breadth; the cost parameter swept and the trades held fixed; the development-selected Sharpe, its deflated value, and the same strategy's later-sample result.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 6 — Move the interpolation peak

> **Break cue:** Deck B, after recording segment S4. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

### Fit exactly, then ask whether the fit predicts

**Purpose:** inspect what happens when the number of coefficients reaches the number of training rows. Each trial has `n=120` training rows, 2,000 independent test rows, and eight signal-bearing features. `X_train_all[:, :p]` keeps the first p columns. `X @ coef` computes predictions; mean squared residuals produce MSE. The test outcomes include noise, unlike the noise-free-teacher error in the earlier textbook figure.

Below the threshold, `solve(X.T @ X + eps*I, X.T @ y)` solves the stabilized least-squares equations. Above it, `X.T @ solve(X @ X.T + eps*I, y)` selects the minimum-length coefficient vector. The tiny `eps` is numerical stabilization, not a validation-selected penalty. There is no fitted intercept here. Many exact fits become possible above the row count, but the fitting rule chooses one.

**Predict, then check:** increase n while retaining values of p on both sides of n. The dangerous region should move with the row count; a coarse grid can miss its peak. Changing `spectrum_decay` also changes signal variation because coefficients stay fixed, so it is not an isolated test of geometry at constant signal strength.

Read the **logarithmic** error axis carefully. The blue training curve displays all values below 0.001 at that floor; the underlying error arrays are unchanged. Twelve trials illustrate one population; the mean near the threshold can be dominated by a few unstable fits. A second descent does not mean the largest model beats the smallest, and choosing a winner from this displayed test curve would require a fresh final test.

In [ ]:
# Edit these three controls, then rerun the cell: n, noise_sd, and spectrum_decay.
n = 120
noise_sd = 0.7
spectrum_decay = 0.0       # 0 = isotropic; larger values create a steeper spectrum
p_grid = np.array([12, 30, 60, 90, 108, 116, 120, 124, 135, 170, 240, 360])
n_trials = 12
p_max = int(p_grid.max())

train_mse = np.zeros((n_trials, len(p_grid)))
test_mse = np.zeros_like(train_mse)
for trial in range(n_trials):
    g = np.random.default_rng(1000 + trial)
    scales = np.exp(-spectrum_decay * np.arange(p_max) / p_max)
    beta = np.zeros(p_max)
    beta[:8] = np.linspace(1.0, 0.25, 8) / np.sqrt(8)
    X_train_all = g.standard_normal((n, p_max)) * scales
    X_test_all = g.standard_normal((2000, p_max)) * scales
    y_train = X_train_all @ beta + noise_sd * g.standard_normal(n)
    y_test = X_test_all @ beta + noise_sd * g.standard_normal(2000)

    for j, p in enumerate(p_grid):
        X_train = X_train_all[:, :p]
        X_test = X_test_all[:, :p]
        # A tiny numerical ridge stabilizes the exact threshold without
        # changing the minimum-norm solution away from singularity.
        eps = 1e-10
        if p <= n:
            coef = np.linalg.solve(
                X_train.T @ X_train + eps * np.eye(p), X_train.T @ y_train
            )
        else:
            coef = X_train.T @ np.linalg.solve(
                X_train @ X_train.T + eps * np.eye(n), y_train
            )
        train_mse[trial, j] = np.mean((y_train - X_train @ coef) ** 2)
        test_mse[trial, j] = np.mean((y_test - X_test @ coef) ** 2)

mean_test = test_mse.mean(axis=0)
peak_ratio = p_grid[np.argmax(mean_test)] / n
print(f"Interpolation peak occurs near p/n = {peak_ratio:.2f}")
print("Change n, noise_sd, or spectrum_decay and see what moves.")

fig, ax = plt.subplots(figsize=(7.5, 3.8))
display_floor = 1e-3
ax.plot(p_grid / n, np.maximum(train_mse.mean(axis=0), display_floor), "o-",
        label="train MSE (floor 0.001)")
ax.plot(p_grid / n, mean_test, "s-", label="held-out MSE")
ax.axvline(1.0, color="0.25", ls="--", lw=1, label="p/n = 1")
ax.set_yscale("log")
ax.set_xlabel("parameter ratio p/n")
ax.set_ylabel("mean squared error (log scale)")
ax.set_title("An exact fit can be unstable on new data")
for j, offset in [(0, (12, 14)), (int(np.argmax(mean_test)), (-95, -8)), (-1, (-70, 14))]:
    ax.annotate(f"test {mean_test[j]:.3f}", (p_grid[j] / n, mean_test[j]),
                xytext=offset, textcoords="offset points", fontsize=9)
ax.set_ylim(display_floor / 2, mean_test.max() * 8)
ax.legend()
plt.tight_layout(); save_lecture_figure("demo6_interpolation_peak.png"); plt.show()

#### After you run

Locate the interpolation threshold on the feature-count axis and describe test
error on both sides of it. Change one declared input—sample size, noise, or
feature count—and state your directional prediction before rerunning.

**Your notes:** `[write here]`

### Mini-project: vary the interpolation threshold

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

Run **Demo 6**. Change one of the declared controls—sample size, spectrum, or
noise level. Record where $p/n=1$ occurs and whether test loss falls again on
held-out independent data.

**Research record:** the control you changed and its range; where $p/n=1$ falls; whether held-out loss declines again past that point.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 7 — Sweep one training-window dimension

> **Optional extension:** This is not a required recorded-lecture break. Predict what the output should show, then run the cell. Record the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

### Compare training windows on the same forecast dates

**Purpose:** balance more history against stale history. Three of 12 coefficients drift smoothly; the simulation makes that truth visible. At forecast time t, `X[t-window:t]` and `y[t-window:t]` stop before t. A Ridge solve fits the window, and `X[t] @ coef` predicts the next held-out outcome. Each candidate starts forecasting at 240, so every row in the comparison faces the same 660 dates.

The objective is sum of squared residuals plus `5 * sum(coef**2)`. The constant 5 stays fixed; relative to **mean** squared error, its penalty is `5 / window`, so the normalized amount of shrinkage changes with window length. The result measures this declared procedure, not a pure effect of memory isolated from all regularization choices.

`tanh(forecasts / 0.5)` makes bounded positions; mean absolute consecutive position change summarizes turnover. Initial entry is omitted in this comparison, unlike Demo 4. MSE and turnover answer different questions; turnover alone is not net profit.

**Try:** predict which window may react faster, then compare both columns. A preferred window on this displayed path is an exploratory choice. Choosing a deployable setting needs development selection and a separate final period.

In [ ]:
# Every candidate faces the same outer dates, features, objective, and position rule.
g = np.random.default_rng(2026)
T, K = 900, 12
X = g.standard_normal((T, K))
beta_t = np.zeros((T, K))
beta_t[:, :3] = np.column_stack([
    0.35 * np.sin(np.arange(T) / 95),
    0.25 * np.cos(np.arange(T) / 130),
    0.20 * np.sin(np.arange(T) / 170 + 0.8),
])
y = np.sum(X * beta_t, axis=1) + 0.65 * g.standard_normal(T)

windows = [30, 60, 120, 240]
first_outer = max(windows)
rows = []
loss_paths = {}
for window in windows:
    forecasts = []
    for t in range(first_outer, T):
        Xw, yw = X[t-window:t], y[t-window:t]
        coef = np.linalg.solve(Xw.T @ Xw + 5.0 * np.eye(K), Xw.T @ yw)
        forecasts.append(X[t] @ coef)
    forecasts = np.asarray(forecasts)
    outcomes = y[first_outer:]
    losses = (outcomes - forecasts) ** 2
    # A fixed bounded mapping keeps the decision comparison identical.
    positions = np.tanh(forecasts / 0.5)
    turnover = np.abs(np.diff(positions)).mean()
    loss_paths[window] = pd.Series(losses).rolling(40).mean().to_numpy()
    rows.append({"window": window, "MSE": losses.mean(), "turnover": turnover})

window_result = pd.DataFrame(rows).set_index("window")
print(window_result.round(3))
print("All rows use the same outer forecast dates:", first_outer, "through", T - 1)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for window in windows:
    axes[0].plot(loss_paths[window], lw=1, label=f"{window}")
axes[0].set_title("Rolling predictive loss")
axes[0].set_xlabel("common outer forecast")
axes[0].set_ylabel("40-period MSE")
axes[0].legend(title="window", ncol=2, fontsize=8)
axes[1].scatter(window_result["MSE"], window_result["turnover"], s=55)
axes[1].margins(x=0.16, y=0.15)
for window, row in window_result.iterrows():
    axes[1].annotate(str(window), (row["MSE"], row["turnover"]), xytext=(4, 4),
                     textcoords="offset points")
axes[1].set_xlabel("predictive MSE")
axes[1].set_ylabel("mean turnover")
axes[1].set_title("One sweep, two ledgers")
plt.tight_layout(); save_lecture_figure("demo7_window_sweep.png"); plt.show()

#### After you run

Record the window selected by predictive loss and whether the turnover result
points to the same window. List the features, outer forecast dates, and other
settings that remained fixed so that the sweep has one interpretation.

**Your notes:** `[write here]`

### Mini-project: sweep one dimension

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

Run **Demo 7**.

1. Hold the features, objective, and common test forecast dates fixed.
2. Sweep only the training-window length.
3. Compare predictive mean squared error (MSE) and mean turnover across windows.

**Research record:** the window lengths tested; the features, objective, and test dates held fixed; predictive MSE and turnover for each window; which window each metric selects; and the limitation of interpreting one seeded path.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 8 — Kalman filter for time-varying beta

> **Guided-reading demo:** Run this with the optional frontier slides. Treat the output as a mechanism check, not as an empirical market result.

### Track a hidden quantity by predicting, then correcting

**Purpose:** estimate a stock's changing sensitivity to the market, called beta. `true_beta` is hidden truth; `stock_ret` is the noisy observation. A Kalman filter carries an estimate and its uncertainty `P` forward. `q` allows beta to move; `r` describes observation noise. These settings are supplied, not learned here; q is deliberately different from the generator's process variance.

**Worked step:** predicted beta 0.5 and a market return of 0.01 imply stock return 0.005. If the observed stock return is 0.007, the surprise is 0.002. The gain `K` determines how much of that surprise changes beta: `beta_pred + K * surprise`. Larger uncertainty about beta usually permits a larger correction, with the observation coefficient and noise held fixed.

The chart compares estimates **after observing today**, including today's data in both the filter and the 60-day regression. Neither current estimate should be presented as a prediction made before that observation. The initial filter estimate is a supplied prior; rolling regression starts only when 60 observations are available.

**Try:** raise q and check whether the filter reacts faster and becomes noisier. The dashed truth line is available only because these data were simulated.

In [ ]:
T = 1000
demo8_rng = np.random.default_rng(608)
true_beta = 0.5 + np.cumsum(demo8_rng.standard_normal(T) * 0.005)
mkt_ret = demo8_rng.standard_normal(T) * 0.01
stock_ret = true_beta * mkt_ret + demo8_rng.standard_normal(T) * 0.005

# Kalman: scalar state beta_t = beta_{t-1} + w, w ~ N(0, q)
#         obs:   stock_ret = beta_t * mkt_ret + v, v ~ N(0, r)
q = 1e-5
r = 0.005 ** 2
beta_hat = np.zeros(T); P = np.zeros(T)
beta_hat[0] = 0.5; P[0] = 0.5
for t in range(1, T):
    # predict
    beta_pred = beta_hat[t-1]
    P_pred = P[t-1] + q
    # update
    H = mkt_ret[t]
    S = H * P_pred * H + r
    K = P_pred * H / S
    beta_hat[t] = beta_pred + K * (stock_ret[t] - H * beta_pred)
    P[t] = (1 - K * H) * P_pred

# Compare estimates AFTER observing day t, using the same information cutoff.
# These are estimates of current beta, not predictions made before day t.
# Compare to a 60-day rolling OLS
roll_beta = np.zeros(T)
W = 60
for t in range(W - 1, T):
    x = mkt_ret[t-W+1:t+1]; y = stock_ret[t-W+1:t+1]
    roll_beta[t] = (x @ y) / (x @ x)
roll_beta[:W - 1] = np.nan

plt.figure(figsize=(8, 3.5))
plt.plot(true_beta, color="k", ls="--", label="true beta", lw=1)
plt.plot(roll_beta, color="gray", label="60-day rolling OLS", lw=0.8)
plt.plot(beta_hat, color="red", label="Kalman", lw=1)
plt.legend(); plt.xlabel("t"); plt.ylabel("beta")
plt.title("Current beta after observing today: filter vs. rolling OLS")
plt.tight_layout(); plt.show()
# Expected: the filtered beta tracks the moving true beta with a lag set by the ratio of
#           process to measurement variance.

#### After you run

Compare the filtered estimate with the planted beta path. Describe the tradeoff
created by the process-variance setting and state why the filtered state remains
an estimate rather than an observed regime.

**Your notes:** `[write here]`

### Demo 9 — A 2-state Gaussian-emission HMM (recoverable case)

> **Guided-reading demo:** Run this with the optional frontier slides. Treat the output as a mechanism check, not as an empirical market result.

### Update probabilities of two hidden volatility states

**Purpose:** track whether today's return likely came from a low- or high-volatility state. `A[i, j]` is the chance of moving from state i to j. `mu` and `sig` supply each state's return distribution. This is **known-parameter filtering**, not HMM parameter fitting: no Baum–Welch or expectation-maximization algorithm runs here.

For each return, `alpha[t-1] @ A` predicts state probabilities before seeing it. `norm.pdf` scores how compatible the return is with each state. Multiplication combines the two, and division by their sum makes the probabilities total one. With prior probabilities `[0.5, 0.5]` and likelihoods proportional to `[1, 3]`, the updated probabilities are `[0.25, 0.75]`.

The orange line uses observations through t. It estimates today's state; next-state probabilities require one more transition step. Smoothing would additionally use later observations and is not implemented. Here state 1 is named high volatility by its supplied sigma, so there is no fitted-label ambiguity.

**Try:** bring the two sigmas closer together. The returns should provide less information about the state. A causal filter is not enough for a deployable pipeline if its parameters were estimated using future observations.

In [ ]:
# Two regimes: low-vol (sigma=1%), high-vol (sigma=3%).
# Self-transition probability 0.98.
T = 2000
A = np.array([[0.98, 0.02], [0.02, 0.98]])
sig = np.array([0.01, 0.03])
mu = np.array([0.0, 0.0])
states = np.zeros(T, dtype=int)
y = np.zeros(T)
demo9_rng = np.random.default_rng(609)
for t in range(1, T):
    states[t] = demo9_rng.choice(2, p=A[states[t-1]])
y = mu[states] + sig[states] * demo9_rng.standard_normal(T)

# Forward filter (causal) for a known-parameter model
def hmm_filter(y, A, mu, sig, pi0=None):
    K = A.shape[0]; T = len(y)
    if pi0 is None: pi0 = np.ones(K) / K
    alpha = np.zeros((T, K))
    em = norm.pdf(y[:, None], loc=mu, scale=sig)
    alpha[0] = pi0 * em[0]; alpha[0] /= alpha[0].sum()
    for t in range(1, T):
        alpha[t] = (alpha[t-1] @ A) * em[t]
        alpha[t] /= alpha[t].sum()
    return alpha

alpha = hmm_filter(y, A, mu, sig)
p_high_filtered = alpha[:, 1]

plt.figure(figsize=(8, 3.5))
plt.plot(p_high_filtered, color="orange", lw=0.7, label="filtered P(high-vol)")
plt.plot(states, color="k", lw=0.4, alpha=0.5, label="true state")
plt.legend(); plt.title("Known-parameter filtering: state probability through today")
plt.xlabel("observation t"); plt.ylabel("probability of high volatility")
plt.tight_layout(); plt.show()
# Known-parameter filtering, not fitted HMM training: sig[1] explicitly names
# the high-volatility state. Estimated-model labels would require alignment.
print("Oracle parameters supplied; this cell filters states and does not learn transitions or emissions.")

#### After you run

State which separation and persistence make the planted states recoverable in
this simulation. Record one reason a fitted state label should not be treated
as an observed economic fact.

**Your notes:** `[write here]`

### Demo 10 — §7 knob-dial: when is a regime recoverable?

> **Guided-reading demo:** Run this with the optional frontier slides. Treat the output as a mechanism check, not as an empirical market result.

### Predictive-density improvement is not proof of state recovery

**Purpose:** compare a two-state oracle's next-observation density with a single Gaussian estimated from the first half of the sample. The grid changes mean separation `d` and state persistence `p`; there is one independent simulated history at each setting. For each test date, compute the predictive mixture **before** updating state probabilities with that date's return. Log density rewards assigning more probability density to what actually occurred; gain is the average log-density difference in natural-log units per observation.

The HMM receives the true transition and emission parameters. This is a favorable oracle reference, not a fitted-model experiment or a guaranteed finite-sample upper bound. The benchmark estimates its mean and standard deviation from training only. Positive gain does not establish accurate state labels, identifiability, economic value, or statistical significance.

**Predict:** at zero mean separation, both emissions are identical and returns cannot distinguish the states. A small nonzero gain can still appear because the single-state benchmark estimates parameters. The dashed zero contour is sample-specific, not a scientific boundary. Repeat seeds and measure state classification separately before making a recoverability claim.

In [ ]:
# Heatmap of OOS log-likelihood gain over a 1-state benchmark on a (Δ, p) grid.
deltas = np.linspace(0.0, 2.5, 6)
ps = np.linspace(0.6, 0.99, 6)
T = 1500
gain = np.zeros((len(ps), len(deltas)))
demo10_rng = np.random.default_rng(610)

for i, p in enumerate(ps):
    A_ = np.array([[p, 1-p], [1-p, p]])
    for j, d in enumerate(deltas):
        mu_ = np.array([-d/2, d/2])
        sig_ = np.array([1.0, 1.0])
        # simulate
        st = np.zeros(T, dtype=int)
        yt = np.zeros(T)
        for t in range(1, T):
            st[t] = demo10_rng.choice(2, p=A_[st[t-1]])
        yt = mu_[st] + sig_[st] * demo10_rng.standard_normal(T)
        ntr = T // 2
        # 1-state benchmark: fit mean and sd on training, score next half
        mu0 = yt[:ntr].mean(); sd0 = yt[:ntr].std()
        ll_bench = norm.logpdf(yt[ntr:], loc=mu0, scale=sd0).mean()
        # 2-state HMM with TRUE parameters (oracle reference, not a finite-sample upper bound):
        alpha_ = hmm_filter(yt[:ntr], A_, mu_, sig_)
        # one-step predictive density on the test block
        cur = alpha_[-1]
        ll_hmm = []
        for tt in range(ntr, T):
            pred = cur @ A_
            em_t = norm.pdf(yt[tt], loc=mu_, scale=sig_)
            ll_hmm.append(np.log((pred * em_t).sum() + 1e-300))
            post = pred * em_t
            cur = post / post.sum()
        gain[i, j] = np.mean(ll_hmm) - ll_bench

fig, ax = plt.subplots(figsize=(6, 4.5))
# Center each colored cell on its actual tested parameter values.
limit = np.max(np.abs(gain))
im = ax.pcolormesh(deltas, ps, gain, shading="nearest",
                   cmap="RdBu", vmin=-limit, vmax=limit)
ax.set_xlabel(r"separation $\Delta$"); ax.set_ylabel("persistence p")
ax.set_yticks(ps); ax.set_yticklabels([f"{value:.2f}" for value in ps])
ax.set_yticks(ps); ax.set_yticklabels([f"{value:.2f}" for value in ps])
ax.set_title("Oracle predictive-density gain: one path per setting")
plt.colorbar(im, ax=ax, label="gain (nats/obs)")
# zero-contour overlay
if gain.min() < 0 < gain.max():
    ax.contour(deltas, ps, gain, levels=[0.0], colors="black", linestyles="--")
plt.tight_layout(); plt.show()

print("Blue means higher test predictive density for the oracle HMM on this one simulated path.")
print("The zero contour is sample-specific; it is not a boundary proving state recoverability.")
print("Parameters were supplied, not fitted. Repeat seeds and assess state recovery separately.")
# At zero separation the two emissions coincide, so returns cannot distinguish states.
# Away from zero, density gain alone is not an identification or state-accuracy test.

#### After you run

Use the heatmap to identify where state recovery improves. Explain separately
what greater emission separation and greater state persistence contribute, and
state why this known-truth result does not validate a market regime label.

**Your notes:** `[write here]`

### Overlapping-returns companion — the same regression the textbook runs

> **Demonstration:** State what you expect before running the cell; afterward, explain which output supports or contradicts that expectation.

### Count reused outcomes when estimating uncertainty

**Purpose:** show why twelve-month returns measured every month do not provide independent rows. The optional Treasury example approximates discount factors from interpolated par yields using an annual-coupon bootstrap. It is a teaching proxy, not an exact traded-bond return reconstruction. `shift(-12)` creates the future endpoint of the target; the forward-spread predictor uses only the current curve.

`lstsq(X, y)` fits the slope once. `resid[:, None] * X` forms each row's regression score. For lags 1 through 11, the code adds lagged score cross-products with decreasing Bartlett weights. Sandwiching their sum between two inverse design matrices gives Newey–West variance estimates. The naive calculation assumes away serial dependence. Changing the lag count changes the uncertainty estimate, not the slope or fitted predictions.

**Worked interpretation:** adjacent twelve-month outcomes share eleven months, so use eleven lags as an overlap-aware starting point; longer dependence may require more. A t-statistic is slope divided by its estimated standard error. The reported effective count is a diagnostic rescaling from the ratio of standard errors, not a literal number of independent bonds.

**Try:** compare zero and eleven lags while holding the regression sample fixed. Explain why the slope stays put while its uncertainty changes; a larger correction is common here, not guaranteed for every dataset.

In [ ]:
# Chapter 15 runs this on the bundled Treasury curve. Here the only knob that
# matters is `lags`: how much overlap the standard error is told about.
def fred_curve(series):
    """Read the bundled Treasury curve locally, or its keyless public source in Colab."""
    relative = "datasets/fred/treasury_curve.csv"
    for root in (Path.cwd(), *Path.cwd().parents):
        local = root / relative
        if local.exists():
            frame = pd.read_csv(local, index_col="date", parse_dates=True)
            return frame[series].dropna(), str(local)
    url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=" + ",".join(series)
    frame = pd.read_csv(url)
    frame = frame.rename(columns={frame.columns[0]: "date"}).set_index("date")
    frame.index = pd.to_datetime(frame.index)
    return frame.apply(pd.to_numeric, errors="coerce").dropna(), url


QUOTED = {1: "DGS1", 2: "DGS2", 3: "DGS3", 5: "DGS5", 7: "DGS7", 10: "DGS10"}
NODES = list(range(1, 11))
curve_raw, curve_source = fred_curve(list(QUOTED.values()))
monthly_curve = curve_raw.resample("ME").last().dropna() / 100.0

# The quoted par curve skips 4, 6, 8 and 9 years. Interpolate annual par yields,
# then use a simplifying annual-coupon bootstrap to obtain discount-factor proxies.
par_yield = pd.DataFrame(index=monthly_curve.index, columns=NODES, dtype=float)
for date, row in monthly_curve.iterrows():
    par_yield.loc[date] = np.interp(NODES, sorted(QUOTED), [row[QUOTED[m]] for m in sorted(QUOTED)])
discount = pd.DataFrame(index=par_yield.index, columns=NODES, dtype=float)
for date, row in par_yield.iterrows():
    earlier = []
    for maturity in NODES:
        discount.loc[date, maturity] = (1 - row[maturity] * sum(earlier)) / (1 + row[maturity])
        earlier.append(discount.loc[date, maturity])
log_discount = np.log(discount)


def overlap_aware(x, y, lags=11):
    """Slope with a Bartlett-weighted Newey-West SE and the textbook OLS SE beside it."""
    X = np.column_stack([np.ones(len(x)), x])
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ beta
    XtX_inv = np.linalg.inv(X.T @ X)
    u = resid[:, None] * X
    S = u.T @ u
    for lag in range(1, lags + 1):
        G = u[lag:].T @ u[:-lag]
        S += (1 - lag / (lags + 1)) * (G + G.T)
    nw = np.sqrt(np.diag(XtX_inv @ S @ XtX_inv))
    naive = np.sqrt(np.diag(XtX_inv * (resid @ resid) / (len(x) - 2)))
    return beta, nw[1], naive[1]


# Twelve-month excess-return proxy against the forward-spot spread that predicts it.
# Sampling monthly makes consecutive rows share eleven of their twelve months.
panels = {}
for n in [2, 3, 5, 7, 10]:
    excess = log_discount[n - 1].shift(-12) - log_discount[n] + log_discount[1]
    forward = log_discount[n - 1] - log_discount[n] + log_discount[1]
    panels[n] = pd.concat([excess.rename("excess"), forward.rename("forward")], axis=1).dropna()

print("Source: Federal Reserve H.15 constant-maturity yields via FRED")
print(f"File: {curve_source}")
print(f"Sample: {par_yield.index.min():%Y-%m} through {par_yield.index.max():%Y-%m}, {len(par_yield)} month-ends")
print("Units: decimal annual yields; target is the next twelve months, sampled every month")
print()
print("  bond   slope   naive t   NW(5)   NW(11)   NW(23)   effective obs at NW(11)")
for n, frame in panels.items():
    x, y = frame["forward"].to_numpy(), frame["excess"].to_numpy()
    beta, _, se_naive = overlap_aware(x, y)
    t_naive = beta[1] / se_naive
    tees = {lag: beta[1] / overlap_aware(x, y, lag)[1] for lag in (5, 11, 23)}
    effective = len(frame) * (tees[11] / t_naive) ** 2
    print(f"  {n:2d}y   {beta[1]:.3f}   {t_naive:7.2f}   {tees[5]:5.2f}   "
          f"{tees[11]:6.2f}   {tees[23]:6.2f}   {effective:6.0f} of {len(frame)}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
lag_grid = range(0, 25)
for n in [2, 5, 10]:
    x, y = panels[n]["forward"].to_numpy(), panels[n]["excess"].to_numpy()
    beta, _, _ = overlap_aware(x, y)
    axes[0].plot(list(lag_grid), [beta[1] / overlap_aware(x, y, lag)[1] for lag in lag_grid],
                 marker="o", ms=3, label=f"{n}y")
axes[0].axhline(2, color="0.3", lw=0.8, ls="--")
axes[0].axvline(11, color="0.5", lw=0.8, ls=":")
axes[0].set_xlabel("Newey-West lags"); axes[0].set_ylabel("t-statistic")
axes[0].set_title("Telling the SE about the overlap")
axes[0].legend(frameon=False, fontsize=8)

five = panels[5]
beta_five, _, _ = overlap_aware(five["forward"].to_numpy(), five["excess"].to_numpy())
axes[1].scatter(five["forward"], five["excess"], s=9, alpha=0.45)
grid = np.linspace(five["forward"].min(), five["forward"].max(), 50)
axes[1].plot(grid, beta_five[0] + beta_five[1] * grid, lw=2.0, color="crimson")
axes[1].axhline(0, color="0.5", lw=0.8, ls=":")
axes[1].set_xlabel("forward-spot spread at t"); axes[1].set_ylabel("excess return, t to t+1y")
axes[1].set_title("The five-year regression")
fig.tight_layout(); plt.show()

print("The slope does not move; only the honesty of its standard error does.")
print("At the correct lag the t-statistics roughly halve and the sample is worth ~1/6 of its row count.")
# Expected: t falls steeply from lag 0, then flattens near the twelve-month
#           overlap. The predictability survives; its significance is smaller
#           than the naive count claimed.

## Where this leaves us

Forecast value depends on more than a headline metric: it depends on effective breadth, dependence, selection, turnover, costs, and stability across changing conditions. A regime label is also an estimated representation, not an observable fact.

Week 7 turns to the numerical and representational machinery behind optimization, dimension reduction, and clustering.